# Kaggle: arxiv only, resume + Parquet
Attach the private dataset containing crawler-code.zip, bundle-info.json, arxiv-seed.zip and s2-seed.zip.
Enable Internet, use CPU, enable the CONTACT_EMAIL secret.
Run this notebook concurrently with the other service, never another arXiv worker.
Use Save Version / Save & Run All. The job stops after about 9 hours to leave time to save output before the session limit.
On a new session, also attach the previous saved notebook output as Input. Do not rely on an unsaved /kaggle/working directory surviving session termination.


In [ ]:
%pip install -q "pyarrow>=15" "python-dotenv>=1.0" "filelock>=3.10"


In [ ]:
from pathlib import Path
import sys
import json
from dataclasses import replace
from kaggle_secrets import UserSecretsClient

bundles = list(Path('/kaggle/input').rglob('bundle-info.json'))
if len(bundles) != 1:
    raise ValueError('Attach exactly one crawler bundle dataset')
INPUT = bundles[0].parent
code_archives = list(INPUT.rglob('crawler-code.zip'))
code_sources = list(INPUT.rglob('src/crawl/split.py'))
if code_archives:
    sys.path.insert(0, str(code_archives[0]))
elif code_sources:
    sys.path.insert(0, str(code_sources[0].parents[2]))
else:
    raise FileNotFoundError('Missing crawler code in bundle')
from src.common.config import get_settings
from src.crawl.split import unpack_cache, import_cache, make_pipeline, run_stage, archive_cache

STAGE = 'arxiv'
ROOT = Path('/kaggle/working') / f'{STAGE}-job'
IDS = json.loads((INPUT / 'bundle-info.json').read_text(encoding='utf-8'))['ids']
MAX_SECONDS = 9 * 3600
MAX_PASSES = 3
RETRY_DELAY = 1800
secrets = UserSecretsClient()
env = {'CONTACT_EMAIL': secrets.get_secret('CONTACT_EMAIL')}
settings = replace(get_settings(env), data_dir=ROOT)
print(STAGE, len(IDS), 'Publication IDs; output:', ROOT)


In [ ]:
# Immutable inputs are copied to this job's writable output; no shared writers.
archives = list(INPUT.rglob(f'{STAGE}-seed.zip'))
archives += sorted(Path('/kaggle/input').rglob(f'{STAGE}-checkpoint.zip'))
cache_roots = []
for index, archive in enumerate(archives):
    seed = Path('/kaggle/working') / f'{STAGE}-input-{index}'
    cache_roots.append(unpack_cache(archive, seed))
# Kaggle may expand ZIP uploads into folders. Both layouts are supported.
cache_roots += [path.parent.parent for path in Path('/kaggle/input').rglob('crawl.jsonl')
                if path.parent.name == 'manifests']
if not cache_roots:
    raise FileNotFoundError('Attach the seed cache or saved notebook checkpoint output')
print('Imported components:', import_cache(cache_roots, ROOT, IDS, STAGE))


In [ ]:
pipeline = make_pipeline(settings, STAGE)
try:
    report = run_stage(pipeline, IDS, STAGE, max_seconds=MAX_SECONDS,
                       max_passes=MAX_PASSES, retry_delay=RETRY_DELAY)
    print({k: v for k, v in report.items() if k != 'pending_ids'})
finally:
    checkpoint = archive_cache(ROOT, Path('/kaggle/working') / f'{STAGE}-checkpoint.zip')
    print('Checkpoint ready for saved notebook output:', checkpoint)


## Saved output
Download the checkpoint ZIP, or attach this saved notebook output to the next session to resume.
The table in `<stage>-job/tables/` is service-specific, not the final complete-Publication dataset.
After both notebooks finish, download arxiv-checkpoint.zip and s2-checkpoint.zip and run locally:

```powershell
.\.venv\Scripts\python.exe -X utf8 -m scripts.merge_split --arxiv arxiv-checkpoint.zip --s2 s2-checkpoint.zip
```
This offline step preserves the original raw/metadata/references/manifest layout and exports
both tables for the intersection of Publications complete in both services. Check merge-report.json for missing IDs.
